In [5]:
# %%
"""
CELL 1 — SETUP
"""
import json
import re
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import pandas as pd

BASE = Path("..")
RAW = BASE / "data" / "ncs_raw_json"
DICT_PATH = BASE / "data" / "dictionary" / "result" / "dictionary_en_ru.json"
OUT_DIR = BASE / "data" / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

GUID_TRIPLE = re.compile(
    r"([A-Za-z0-9_.'/\-]{2,120}),\s*([0-9A-Fa-f]{32}),\s*([^\n\r\"]{2,200})"
)

def jload(p: Path):
    with open(p, encoding="utf-8") as f:
        return json.load(f)

def clean(s: str) -> str:
    s = re.sub(r"\{[^}]*\}", "", s)
    s = re.sub(r"\[/?[^\]]*\]", "", s)
    s = re.sub(r"\s+", " ", s).strip(" -:\n\t")
    return s

print("DICT:", DICT_PATH.resolve(), "exists:", DICT_PATH.exists())

DICT: /var/home/nexpg/RawBL4ToCutBDInterpreter/data/dictionary/result/dictionary_en_ru.json exists: True


In [6]:
# %%
"""
CELL 2 — DICTIONARY
"""
with open(DICT_PATH, encoding="utf-8") as f:
    raw_dict = json.load(f)

guid_map = {}
en_map = defaultdict(list)

for k, v in raw_dict.items():
    if not isinstance(v, dict):
        continue
    en = (v.get("en") or "").strip()
    ru = (v.get("ru") or "").strip()
    entry = {"en": en, "ru": ru}
    ku = str(k).replace("-", "").upper()
    guid_map[ku] = entry
    guid_map[str(k).lower()] = entry
    if en:
        en_map[en.lower()].append(ru if ru else None)

def translate_by_guid(guid: str | None) -> str:
    if not guid:
        return "—"
    g = str(guid).replace("-", "").upper()
    hit = guid_map.get(g) or guid_map.get(str(guid).lower())
    if not hit:
        return "(перевод не найден)"
    ru = (hit.get("ru") or "").strip()
    return ru if ru else "(перевод не найден)"

def translate_by_en(text: str | None) -> str:
    if not text or text == "—":
        return "—"
    variants = [r for r in en_map.get(text.lower(), []) if r]
    if not variants:
        return "(перевод не найден)"
    uniq = list(dict.fromkeys(variants))
    if len(uniq) > 1:
        return "(требуется ручная проверка)"
    return uniq[0]

print(f"guid_map={len(guid_map)}")

guid_map=232656


In [7]:
# %%
"""
CELL 3 — SOURCE A: part_stat keys from inv enhancement deps
         SOURCE B: uistat_enh_stat_* labels + GUID
         JOIN by suffix (без whitelist слов)
"""
# --- A: part keys ---
stat_keys = set()
for path in sorted(p for p in RAW.rglob("inv*.json") if "name" not in p.name.lower()):
    try:
        data = jload(path)
    except Exception:
        continue
    for tbody in (data.get("tables") or {}).values():
        for rec in tbody.get("records") or []:
            for entry in rec.get("entries") or []:
                it = str(entry.get("key") or "").lower()
                if "enhancement" not in it:
                    continue
                for dep in entry.get("dep_entries") or []:
                    dtn = str(dep.get("dep_table_name") or "").lower()
                    if dtn not in ("stat_group1", "stat_group2", "stat_group3"):
                        continue
                    dkey = str(dep.get("key") or "").lower()
                    if dkey.startswith("part_stat"):
                        stat_keys.add(dkey)
                    val = dep.get("value") if isinstance(dep.get("value"), dict) else {}
                    pairs = ((val.get("parttypeselectionrules") or {}).get("pairs") or {})
                    if not isinstance(pairs, dict):
                        continue
                    for pair in pairs.values():
                        if not isinstance(pair, dict):
                            continue
                        if str(pair.get("key") or "").lower() not in (
                            "stat_group1", "stat_group2", "stat_group3"
                        ):
                            continue
                        for p in (pair.get("value") or {}).get("parts") or []:
                            part = p.get("part") if isinstance(p, dict) else p
                            if part and str(part).lower().startswith("part_stat"):
                                stat_keys.add(str(part).lower())

print(f"part_stat keys: {len(stat_keys)}")

# --- B: labels ---
label_by_suffix = {}
for path in sorted(RAW.rglob("*.json")):
    try:
        text = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue
    if "uistat_enh_stat_" not in text.lower() and "enhancement_uistats" not in text.lower():
        continue
    for m in GUID_TRIPLE.finditer(text):
        cat, guid, disp = m.group(1).strip(), m.group(2).upper(), m.group(3).strip()
        name = clean(disp)
        if not name or len(name) < 2 or len(name) > 100:
            continue
        win = text[max(0, m.start() - 300): m.start() + 80]
        keys = re.findall(r"uistat_enh_stat_[a-z0-9_]+", win, re.I)
        if not keys and "enhancement_uistats" not in cat.lower():
            continue
        uk = keys[-1].lower() if keys else None
        if not uk:
            continue
        suf = uk.replace("uistat_enh_stat_", "")
        label_by_suffix[suf] = {"name_eng": name, "guid": guid, "uistat_key": uk}

print(f"uistat labels: {len(label_by_suffix)}")

# --- JOIN ---
def suffixes_from_part(pk: str) -> list[str]:
    p = pk.lower()
    for pref in ("part_stat3_", "part_stat2_", "part_stat_"):
        if p.startswith(pref):
            return [p[len(pref):]]
    return []

uniq = {}  # name_lower → {name_eng, guid}
for pk in sorted(stat_keys):
    hit = None
    for suf in suffixes_from_part(pk):
        hit = label_by_suffix.get(suf)
        if hit:
            break
        for k, v in label_by_suffix.items():
            if k == suf or k.endswith(suf) or suf.endswith(k):
                hit = v
                break
        if hit:
            break
    if not hit:
        continue
    n = re.sub(r"\s+is increased by\s*$", "", hit["name_eng"], flags=re.I).strip()
    sig = n.lower()
    if sig not in uniq:
        uniq[sig] = {"name_eng": n, "guid": hit["guid"]}

print(f"unique bonuses: {len(uniq)}")

part_stat keys: 216
uistat labels: 72
unique bonuses: 62


In [8]:
# %%
"""
CELL 4 — DF + SAVE  (только name_eng, name_ru)
"""
rows = []
for v in sorted(uniq.values(), key=lambda x: x["name_eng"].lower()):
    eng = v["name_eng"]
    ru = translate_by_guid(v["guid"])
    if ru in ("(перевод не найден)", "—"):
        ru = translate_by_en(eng)
    rows.append({"name_eng": eng, "name_ru": ru})

df = pd.DataFrame(rows).drop_duplicates(subset=["name_eng"]).reset_index(drop=True)
print(df.shape)
print(df.to_string())

ts = datetime.now().strftime("%m-%d-%Y_%I-%M-%S%p")
out = OUT_DIR / f"bl4_enhancement_stat_bonuses_{ts}.csv"
df.to_csv(out, index=False, encoding="utf-8-sig", sep=";", quoting=1)
print("Wrote:", out.resolve())

(62, 2)
                         name_eng                                                                                                            name_ru
0          Assault Rifle Accuracy                               [secondary]Точность автоматов[/secondary] повышается на [secondary]{mod}[/secondary]
1   Assault Rifle ADS Proficiency                   [secondary]Навык прицеливания из автомата[/secondary] повышается на [secondary]{mod}[/secondary]
2   Assault Rifle Critical Damage                    [secondary]Критический урон от автоматов[/secondary] повышается на [secondary]{mod}[/secondary]
3            Assault Rifle Damage                                [secondary]Урон от автоматов[/secondary] повышается на [secondary]{mod}[/secondary]
4       Assault Rifle Equip Speed                 [secondary]Скорость смены оружия на автомат[/secondary] повышается на [secondary]{mod}[/secondary]
5         Assault Rifle Fire Rate                       [secondary]Скорострельность автоматов[/sec